# 03 - 训练循环与优化器

本 notebook 深入讲解 `trainer/train_sft_omni.py`，包括：
1. 启动参数与配置解析
2. 模型初始化与 DDP 包装
3. `compute_batch_loss`：text loss、audio loss、RVQ 权重、global loss norm
4. `MuonWithAuxAdam` 优化器
5. 学习率调度
6. checkpoint / resume 机制
7. 验证与 metrics
8. 训练中的已知问题与调试方法

## 1. 启动参数概览

`train_sft_omni.py` 有近 60 个命令行参数，可分成几类：
- 模型架构：`hidden_size`, `num_hidden_layers`, `talker_*`, `bridge_layer`, `use_moe`
- 训练流程：`epochs`, `batch_size`, `learning_rate`, `muon_lr`, `accumulation_steps`, `grad_clip`
- 数据：`data_path`, `max_seq_len`, `rank_sharded_data`
- 训练模式：`mode`（all / audio_proj / vision_proj）, `freeze_backbone`
- Phase-0 修复：`warmup_ratio`, `loss_norm`, `rvq_layer_weights`, `audio_stop_weight`, `val_data_path`, `val_interval`, `finite_guard`
- 工程：`use_compile`, `gradient_checkpointing`, `ddp_broadcast_buffers`, `use_wandb`

这些参数由 `scripts/run_full_train_muon_*.sh` 包装器通过环境变量注入。

In [ ]:
import os, sys
import subprocess
ROOT = "/home/genesis/Projects/minimind-o"
os.chdir(ROOT)
sys.path.insert(0, ROOT)

# 查看所有参数帮助（通过 subprocess 调用，避免 notebook magic 语法）
result = subprocess.run(
    ["python", "trainer/train_sft_omni.py", "--help"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 2. 模型初始化与权重加载

`init_omni_model`（`trainer/trainer_utils.py:71-110`）负责：
1. 加载 tokenizer
2. 构造 `MiniMindOmni`
3. 如果 `--from_weight` 不是 none，加载对应 `.pth`
4. 如果talker未训练过且 `talker_hidden_size == hidden_size`，从 thinker 最后几层复制初始化 talker
5. 根据 `freeze_backbone` 冻结参数

In [ ]:
from trainer.trainer_utils import init_omni_model
from model.model_omni import OmniConfig

cfg = OmniConfig(hidden_size=768, num_hidden_layers=8, num_attention_heads=8, num_key_value_heads=4)

# 不加载权重，只初始化
model, tokenizer = init_omni_model(cfg, from_weight='none', save_dir='out', device='cpu')
print(f"model type: {type(model).__name__}")
print(f"tokenizer vocab: {len(tokenizer)}")

## 3. compute_batch_loss 详解

代码见 `trainer/train_sft_omni.py:39-106`。

### 3.1 text loss
- 使用 `CrossEntropyLoss(reduction='none')` 逐 token 计算。
- mask 掉 `-100`（非 assistant 区域）。
- `loss_norm=global` 时，跨 DDP 聚合总有效 token 数，再全局平均。

### 3.2 audio loss
- 对 8 层 RVQ 分别计算 CE。
- 每层可独立加权（`rvq_layer_weights`）。
- stop token（2050）乘以 `audio_stop_weight` 的额外权重。

### 3.3 global vs local loss norm
- `local`：每个 rank 用自己的有效 token 数平均。
- `global`：所有 rank 的有效 token 数求和，再平均。乘以 `world_size` 是为了保持每个 rank 上 loss 的梯度贡献与 global 一致。

In [ ]:
import torch
import torch.nn as nn
from types import SimpleNamespace
from trainer.train_sft_omni import compute_batch_loss, parse_rvq_layer_weights

# 构造 dummy output
B, T, V_text, V_audio = 2, 16, 6400, 2112
res = SimpleNamespace(
    logits=torch.randn(B, T, V_text),
    audio_logits=[torch.randn(B, T, V_audio) for _ in range(8)],
    aux_loss=torch.tensor(0.0),
)
labels = torch.randint(0, V_text, (B, T))
audio_labels = torch.randint(0, V_audio, (B, 8, T))
loss_fct = nn.CrossEntropyLoss(reduction='none')

# local loss norm
text_loss, audio_loss = compute_batch_loss(
    res, labels, audio_labels, loss_fct,
    rvq_weights=(1.0,)*8, audio_stop_weight=10.0, loss_norm='local'
)
print(f"local  text_loss: {text_loss.item():.4f}, audio_loss: {audio_loss.item():.4f}")

# 观察 RVQ 加权效果
non_uniform = parse_rvq_layer_weights("2,1.5,1.2,1,0.8,0.7,0.6,0.5")
text_loss2, audio_loss2 = compute_batch_loss(
    res, labels, audio_labels, loss_fct,
    rvq_weights=non_uniform, audio_stop_weight=10.0, loss_norm='local'
)
print(f"non-uniform audio_loss: {audio_loss2.item():.4f}")

## 4. MuonWithAuxAdam 优化器

代码见 `trainer/optimizers.py:39-185`。

### 4.1 参数分组规则
`_is_muon_parameter` 决定哪些参数走 Muon：
- 必须是 2D 且 requires_grad
- 名称中不含 `embed`, `lm_head`, `head`, `norm`, `bias`, `audio_encoder`, `vision_encoder`

### 4.2 Muon 更新
- 对梯度做 Newton-Schulz 迭代求近似正交矩阵。
- 乘以 `max(1, m/n)^0.5` 缩放。
- 使用 Nesterov momentum。

### 4.3 AdamW fallback
- embedding / norm / bias / 投影层等非 2D 参数用 AdamW。
- fallback 的学习率使用 `args.learning_rate`，不受 `muon_lr` 影响。

In [ ]:
from trainer.optimizers import build_optimizer, _is_muon_parameter
from types import SimpleNamespace

# 统计模型中 Muon / AdamW 参数数量
muon_params, adam_params = [], []
for n, p in model.named_parameters():
    if p.requires_grad:
        if _is_muon_parameter(n, p):
            muon_params.append(p)
        else:
            adam_params.append(p)
print(f"Muon params : {sum(p.numel() for p in muon_params)/1e6:.2f} M")
print(f"AdamW params: {sum(p.numel() for p in adam_params)/1e6:.2f} M")

## 5. 学习率调度

代码见 `trainer/trainer_utils.py:26-34`。

公式：
- warmup 阶段：`lr * (step / warmup_steps)`
- warmup 后：`lr * (0.1 + 0.45 * (1 + cos(pi * progress)))`，从 `0.1 + 0.45*2 = 1.0 * lr` 降到 `0.1 * lr`。

注意：每个 param_group 还会乘以 `lr_scale`，用于让 Muon group 使用不同的有效学习率。

In [ ]:
import matplotlib.pyplot as plt
from trainer.trainer_utils import get_lr

total_steps = 1000
warmup = 100
base_lr = 5e-4
lrs = [get_lr(s, total_steps, base_lr, warmup) for s in range(total_steps)]

plt.plot(lrs)
plt.xlabel("step")
plt.ylabel("lr")
plt.title("MiniMind-O LR schedule")
plt.grid(True)
plt.show()

## 6. checkpoint / resume 机制

代码见 `trainer/trainer_utils.py:113-169`。

保存两种 checkpoint：
- `out/{weight}_{hidden}.pth`：半精度模型权重，用于推理/继续训练加载。
- `checkpoints/{weight}_{hidden}_resume.pth`：全精度模型 + optimizer + scaler + epoch/step + wandb_id。

### resume 时的关键修复
代码见 `trainer/train_sft_omni.py:600-611`：
```python
raw_model = getattr(model, '_orig_mod', model)
load_result = raw_model.load_state_dict(ckp_data['model'], strict=False)
```

**为什么重要**：如果 `model` 被 `torch.compile` 包装，直接 `model.load_state_dict` 会把权重加载到 compiled wrapper，大量真实权重被静默跳过，resume 后 loss 会暴涨。当前代码已修复。

## 7. DDP 与 rank-sharded data

### DDP broadcast_buffers
代码见 `trainer/train_sft_omni.py:614-616`。

Transformer 中的 LayerNorm / RMSNorm 的 `weight` 是 buffer（如果未设为 parameter），`broadcast_buffers=True` 会在每次 forward 前同步所有 rank 的 buffers。
在 MiniMind-O 中，RMSNorm weight 是 `nn.Parameter`，但 DDP 的 buffer 同步仍可能触发不必要的 NCCL 通信。

**Run C 的关键修复**：`DDP_BROADCAST_BUFFERS=0`，否则 Stage1 会出现 NCCL buffer-broadcast timeout。

## 8. 验证流程

`evaluate_loader`（`trainer/train_sft_omni.py:146-215`）每 `val_interval` step 跑一次验证集。

输出：
- `loss / text / audio`：全局平均
- `rvq_layer_losses`：8 层各自 CE
- `rvq_stop_counts / rvq_stop_rates`：每层 stop token 占比

**注意**：
- 验证也会跑 `compute_batch_loss`，因此 val 曲线可以直接反映训练 health。
- `finite_guard` 在验证阶段检查 `val_loss` 非有限。
- 验证集 `dataset/_val/` 是从训练数据中采样的，用于监控而非严格 held-out generalization。

## 9. 训练循环中的问题清单

| 问题 | 位置 | 影响 | 建议 |
| --- | --- | --- | --- |
| resume 加载 compiled model 的 `_orig_mod` | `train_sft_omni.py:601` | 已修复，但老 checkpoint 仍可能不兼容 | 始终使用修复后的 resume 路径 |
| DDP broadcast_buffers 默认 1 | `train_sft_omni.py:468` | NCCL timeout / 性能下降 | v3/v4 强制设为 0 |
| `loss_norm=global` 乘以 world_size | `compute_batch_loss:52,79` | 保持梯度贡献，但数值上不是普通平均 | 与 PyTorch 默认 DDP 行为不同，需注意 |
| gradient accumulation 只在整除时更新 | `train_sft_omni.py:325-330` | 若 iters 不是 accum 倍数，最后一段梯度会被丢弃 | epoch 末尾有补偿逻辑（404-408） |
| `max_train_steps` 与 `save_interval` 可能冲突 | `train_sft_omni.py:639-647` | early stop 时可能不保存最后权重 | 注意 checkpoint 管理 |
| `SkipBatchSampler` 随机种子每个 epoch 重新设置 | `train_sft_omni.py:632` | rank-sharded 数据与 DDP sampler 的随机性需一致 | 检查各 rank 是否同分布 |
| `evaluate_loader` 使用全局变量 `args` 和 `model` | `train_sft_omni.py:148-149` | notebook/单元测试不方便复用 | 可重构为纯函数 |
| wandb 实际是 swanlab | `train_sft_omni.py:533` | 代码里 `import swanlab as wandb`，不要混用 | 注意 api 差异 |

## 10. 最小训练 smoke

下面的命令使用 calibration 数据跑一个极短训练，只验证训练循环能跑通。
不要在大数据上运行。

In [ ]:
calib_parquet = "dataset/_calib/sft_t2a_144rows.parquet"
if os.path.exists(calib_parquet):
    cmd = (
        f"CUDA_VISIBLE_DEVICES=0 python trainer/train_sft_omni.py "
        f"--data_path {calib_parquet} "
        f"--from_weight none --epochs 1 --batch_size 4 --max_train_steps 4 "
        f"--max_seq_len 128 --save_weight smoke_nb03 --save_dir out/_smoke "
        f"--log_interval 1 --use_compile 0"
    )
    print(cmd)
    print("\n请在终端手动运行上述命令，验证训练循环。\n注意：这会创建 out/_smoke/ 目录。")
else:
    print(f"calibration parquet 不存在: {calib_parquet}")